In [1]:
# ============================================================
# Baseline experiment
#
# BERT + LoRA
# No bias mitigation
# ============================================================


# ============================================================
# SETUP
# ============================================================

# Only needed in Google Colab:
# %cd /content/gender-bias-hate-speech
# !pip install -r requirements.txt



In [2]:
from pathlib import Path
import os

PROJECT_ROOT = Path(r"C:\Users\lunaj\Desktop\gender-bias-hate-speech")
os.chdir(PROJECT_ROOT)

print(f"Project root: {Path.cwd()}")

Project root: C:\Users\lunaj\Desktop\gender-bias-hate-speech


In [3]:
%pip install peft

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
# 2. IMPORTS
# ============================================================

import json
from pathlib import Path

import numpy as np
import pandas as pd

from src.config import ExperimentConfig, prepare_run

from src.data.dataset import load_data_from_config, prepare_dataset, split_dataset
from src.data.text_processing import deep_clean_tweet
from src.data.gender_labeling import add_gender_label, count_label_distribution

from src.training.model import load_model, load_tokenizer, tokenize_dataset, load_fine_tuned_model, prepare_lora
from src.training.train import (
    create_training_arguments,
    create_trainer,
)

from src.evaluation.metrics import classification_metrics
from src.evaluation.fairness import equalized_odds_gaps

c:\Users\lunaj\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
# 3. LOAD CONFIGURATION
# ============================================================

config = ExperimentConfig.from_files(
    model="configs/model.yaml",
    training="configs/training.yaml",
    evaluation="configs/evaluation.yaml",
    dataset="configs/datasets/davidson_multi.yaml",
    mitigations=[
        "configs/mitigation/baseline.yaml",
    ],
)

print("Resolved configuration:")
print(config.to_dict())


Resolved configuration:
{'model': {'name': 'bert-base-uncased', 'is_fine_tuned': False}, 'lora': {'enabled': True, 'r': 8, 'alpha': 16, 'dropout': 0.1, 'target_modules': ['query', 'value']}, 'training': {'learning_rate': 2e-05, 'batch_size': 16, 'epochs': 3, 'weight_decay': 0.01, 'warmup_ratio': 0.0, 'seed': 42}, 'evaluation': {'batch_size': 32, 'metrics': ['accuracy', 'f1', 'tpr_gap', 'equalized_odds'], 'counterfactual_testing': True}, 'dataset': {'name': 'davidson', 'config_name': None, 'text_column': 'text', 'label_column': 'label', 'gender_column': 'gender_label', 'train_split': 'train', 'validation_split': 'validation', 'test_split': 'test', 'source': 'local', 'kaggle': {'data_path': 'eldrich/hate-speech-offensive-tweets-by-davidson-et-al', 'file_path': 'data/labeled_data.csv'}, 'local': {'path': 'data/raw/davidson_data.csv'}, 'columns': {'text': 'tweet', 'label': 'class'}, 'label_mapping': {'mode': 'multiclass', 'num_labels': 3, 'mapping': {0: 0, 1: 1, 2: 2}, 'names': {0: 'hate_s

In [6]:
# 4. CREATE RUN
# ============================================================

config, run_dir = prepare_run(
    experiment_name="baseline_dataset_a",
    config=config,
)

print(f"Run directory: {run_dir}")


Run directory: results\baseline_dataset_a\run_028


In [7]:
# 5. LOAD DATASET
# ============================================================

dataset = load_data_from_config(config)
dataset = prepare_dataset(dataset, config)

print(dataset)



Map: 100%|██████████| 24783/24783 [00:01<00:00, 19854.94 examples/s]

Dataset({
    features: ['text', 'label', 'category'],
    num_rows: 24783
})


In [8]:
# 6. Clean dataset // text preprocessing
# ============================================================

dataset.map(deep_clean_tweet)



Map: 100%|██████████| 24783/24783 [00:01<00:00, 17933.33 examples/s]


Dataset({
    features: ['text', 'label', 'category'],
    num_rows: 24783
})

In [9]:
# 7. add Gender Label
# ============================================================
dataset = dataset.map(add_gender_label)

print("Gender label distribution:")
count_label_distribution(dataset, config)

### add more analysis....


Map: 100%|██████████| 24783/24783 [00:01<00:00, 23907.10 examples/s]


Gender label distribution:
           ['hate_speech', 'offensive', 'neither']
male       [159, 1589, 431]
female     [69, 2276, 191]
equal      [11, 185, 21]
none       [1191, 15140, 3520]


[[159, 1589, 431], [69, 2276, 191], [11, 185, 21], [1191, 15140, 3520]]

In [10]:
# Split dataset 
dataset = split_dataset(dataset, config)


In [11]:
# 7. Preprocessing MITIGATION
# ============================================================

mitigation_config = config.get(
    "mitigation",
    {}
)

print("\nMitigation configuration:")
print(mitigation_config)




Mitigation configuration:
{}


In [12]:
# ============================================================
# 6. DATASET SANITY CHECKS
# ============================================================

print("\nDataset splits:")

for split in dataset.keys():
    print(f"{split}: {len(dataset[split])}")


print("\nColumns:")
print(dataset["train"].column_names)


print("\nFirst training example:")
print(dataset["train"][0])


# Target label distribution
if "label" in dataset["train"].column_names:

    print("\nLabel distribution:")

    print(
        dataset["train"]
        .to_pandas()["label"]
        .value_counts()
        .sort_index()
    )


# Gender-label distribution
if "gender_label" in dataset["train"].column_names:

    print("\nGender distribution:")

    print(
        dataset["train"]
        .to_pandas()["gender_label"]
        .value_counts()
        .sort_index()
    )




Dataset splits:
train: 19826
validation: 2478
test: 2479

Columns:
['text', 'label', 'category', 'gender_label']

First training example:
{'text': 'I think I just gotta accept the fact that all these bitches gonna be hoes til they around the age of 27 and life will be a lot easier', 'label': 1, 'category': 'offensive', 'gender_label': 3}

Label distribution:
label
0     1138
1    15385
2     3303
Name: count, dtype: int64

Gender distribution:
gender_label
0     1749
1     1995
2      170
3    15912
Name: count, dtype: int64


In [13]:
# ============================================================
# 8. LOAD TOKENIZER
# ============================================================

tokenizer = load_tokenizer(
    config["model"]["name"]
)

print("\nTokenizer loaded:")
print(tokenizer.__class__.__name__)






Tokenizer loaded:
BertTokenizer


In [14]:
# ============================================================
# 9. TOKENIZE DATASET
# ============================================================

tokenized_dataset = tokenize_dataset(
    dataset=dataset,
    tokenizer=tokenizer,
    
)

print("\nTokenized dataset:")
print(tokenized_dataset)

Map: 100%|██████████| 2479/2479 [00:00<00:00, 8051.44 examples/s]


Tokenized dataset:
DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'category', 'gender_label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 19826
    })
    validation: Dataset({
        features: ['text', 'label', 'category', 'gender_label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 2478
    })
    test: Dataset({
        features: ['text', 'label', 'category', 'gender_label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 2479
    })
})


In [15]:
# ============================================================
# 10. LOAD MODEL
# ============================================================

# load_model() should:
#
# 1. load bert-base-uncased
# 2. create the sequence-classification head
# 3. apply LoRA
#
# LoRA is part of the standard model setup and therefore
# should happen inside src/training/model.py.


label_names = {
    int(key): value
    for key, value in config["dataset"]["label_mapping"]["names"].items()
}

#if fine-tuned model exists, load it. Otherwise, load the base model and prepare it for LoRA fine-tuning.
is_fine_tuned = config["model"].get("is_fine_tuned", False)

if is_fine_tuned:
    model = load_fine_tuned_model(
        model_path=config["model"]["fine_tuned_path"],
        num_names=label_names
    )
else:


    model = load_model(
        model_name=config["model"]["name"],
        label_names=label_names,
    )

    lora_model = prepare_lora(
        model=model,
        num_labels=len(label_names),
        lora_r=config["lora"]["r"],
        lora_alpha=config["lora"]["alpha"],
        lora_dropout=config["lora"]["dropout"],
    )

print("\nModel loaded:")
print(lora_model.__class__.__name__)


# Check that LoRA is actually active.
if hasattr(lora_model, "print_trainable_parameters"):

    print("\nTrainable parameters:")

    lora_model.print_trainable_parameters()




Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7340.95it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoi


Model loaded:
PeftModelForSequenceClassification

Trainable parameters:
trainable params: 297,219 || all params: 109,781,766 || trainable%: 0.2707


In [16]:
# ============================================================
# 11. TRAINING ARGUMENTS
# ============================================================

training_args = create_training_arguments(
    config=config,
    output_dir = run_dir / "checkpoints",
)

print("\nTraining arguments:")
print(training_args)



KeyError: 'num_train_epochs'

In [ ]:
# ============================================================
# 12. CREATE TRAINER
# ============================================================

trainer = create_trainer(
    model=model,
    tokenizer=tokenizer,
    training_args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
)

print("\nTrainer created.")



In [ ]:
# ============================================================
# 13. TRAIN MODEL
# ============================================================

train_result = trainer.train()


print("\nTraining finished.")

print("\nTraining metrics:")
print(train_result.metrics)


In [ ]:
# ============================================================
# 14. TEST SET PREDICTIONS
# ============================================================

prediction_output = trainer.predict(
    tokenized_dataset["test"]
)


logits = prediction_output.predictions

y_true = prediction_output.label_ids

y_pred = np.argmax(
    logits,
    axis=1,
)


print("\nNumber of predictions:")
print(len(y_pred))


In [ ]:

# ============================================================
# 15. PERFORMANCE METRICS
# ============================================================

performance = classification_metrics(
    y_true=y_true,
    y_pred=y_pred,
)

print("\nPerformance metrics:")

for name, value in performance.items():
    print(f"{name}: {value}")


In [ ]:
# ============================================================
# 16. FAIRNESS METRICS
# ============================================================

fairness = {}


if "gender_label" in tokenized_dataset["test"].column_names:

    gender_labels = tokenized_dataset["test"]["gender_label"]

    fairness = equalized_odds_gaps(
        y_true=y_true,
        y_pred=y_pred,
        group_labels=gender_labels,
    )

    print("\nFairness metrics:")

    for name, value in fairness.items():
        print(f"{name}: {value}")

else:

    print(
        "\nNo gender_label column found. "
        "Fairness evaluation skipped."
    )


In [ ]:
# ============================================================
# 17. SAVE INDIVIDUAL PREDICTIONS
# ============================================================

prediction_data = {
    "text": dataset["test"]["text"],
    "label": y_true,
    "prediction": y_pred,
}


if "gender_label" in dataset["test"].column_names:

    prediction_data["gender_label"] = (
        dataset["test"]["gender_label"]
    )


predictions_df = pd.DataFrame(
    prediction_data
)


predictions_path = (
    run_dir / "predictions.csv"
)


predictions_df.to_csv(
    predictions_path,
    index=False,
)


print(
    f"\nPredictions saved to: "
    f"{predictions_path}"
)


In [ ]:
# ============================================================
# 18. SAVE METRICS
# ============================================================

metrics = {
    "performance": performance,
    "fairness": fairness,
}


metrics_path = (
    run_dir / "metrics.json"
)


with open(
    metrics_path,
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        metrics,
        file,
        indent=2,
    )


print(
    f"Metrics saved to: "
    f"{metrics_path}"
)



In [ ]:
# ============================================================
# 19. OPTIONAL: SAVE FINAL MODEL
# ============================================================

model_dir = (
    run_dir / "model"
)


trainer.save_model(
    model_dir
)


tokenizer.save_pretrained(
    model_dir
)


print(
    f"Model saved to: "
    f"{model_dir}"
)


In [ ]:
# ============================================================
# 20. FINAL SUMMARY
# ============================================================

print("\n")
print("=" * 60)
print("BASELINE EXPERIMENT FINISHED")
print("=" * 60)

print(
    f"Run directory: "
    f"{run_dir}"
)

print("\nPerformance:")

for name, value in performance.items():

    print(
        f"  {name}: "
        f"{value:.4f}"
    )


if fairness:

    print("\nFairness:")

    for name, value in fairness.items():

        print(
            f"  {name}: "
            f"{value:.4f}"
        )


print("\nSaved files:")

print(
    f"  Config: "
    f"{run_dir / 'config.yaml'}"
)

print(
    f"  Metrics: "
    f"{metrics_path}"
)

print(
    f"  Predictions: "
    f"{predictions_path}"
)

print(
    f"  Model: "
    f"{model_dir}"
)

print("=" * 60)